In [26]:
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.io as pio
from pathlib import Path
from scipy.stats import gaussian_kde
pio.renderers.default = "notebook_connected"

In [27]:
base_dir = Path.cwd()
data_path = (base_dir/'data'/'airbnb_london.csv')
a = pd.read_csv(data_path)

In [28]:
p95 = a['price'].quantile(0.95)
a_cap = a.loc[a['price'] <= p95].copy()
print(f"95th percentile price: £{p95:.0f}")
print(a_cap.groupby('room_type')['price'].describe().round(1))

95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


In [29]:
a_t = a_cap.loc[a_cap['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

med_entire  = a_t.loc[a_t['room_type'] == 'Entire home/apt']['price'].median()
med_private = a_t.loc[a_t['room_type'] == 'Private room']['price'].median()

palette = {'Entire home/apt': '#2E75B6', 'Private room': '#E07B39'}

fig = px.histogram(
    a_t,
    x='price',
    color='room_type',
    barmode='overlay',
    nbins=60,
    opacity=0.6,                               
    color_discrete_map=palette,
    labels={'price': 'Nightly Price (£)', 'room_type': 'Room Type'},
    title='Entire homes cost roughly twice as much as private rooms — distributions barely overlap',
)

fig.add_vline(
    x=med_entire, line_dash='dash', line_color='#2E75B6', line_width=1.5,
    annotation=dict(
        text=f'Entire median £{med_entire:.0f}',
        font=dict(color='#2E75B6', size=11),
        xanchor='left', yanchor='top', xshift=8,  
    ),
)

fig.add_vline(
    x=med_private, line_dash='dash', line_color='#E07B39', line_width=1.5,
    annotation=dict(
        text=f'Private median £{med_private:.0f}',
        font=dict(color='#E07B39', size=11),
        xanchor='right', yanchor='top', xshift=-8,  
    ),
)

fig.add_annotation(
    xref='paper', yref='paper', x=0.98, y=0.95,
    text=f'Prices capped at 95th percentile (£{p95:.0f})',
    showarrow=False,
    font=dict(size=10, color='#888888'),
    align='right',
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(title='Room Type', orientation='h', y=1.08),
    margin=dict(l=60, r=40, t=70, b=40),
)
fig.update_xaxes(gridcolor='#EEEEEE')
fig.update_yaxes(gridcolor='#EEEEEE', title='Number of Listings')

fig.show(renderer="browser")

In [30]:
entire  = a_t.loc[a_t['room_type'] == 'Entire home/apt']['price']
private = a_t.loc[a_t['room_type'] == 'Private room']['price']

x_range = np.linspace(0, a_t['price'].max(), 300)

kde_entire  = gaussian_kde(entire)(x_range)
kde_private = gaussian_kde(private)(x_range)

df_kde = pd.DataFrame({
    'price':     np.tile(x_range, 2),
    'density':   np.concatenate([kde_entire, kde_private]),
    'room_type': ['Entire home/apt'] * 300 + ['Private room'] * 300,
})

fig = px.line(
    data_frame=df_kde,
    x='price',
    y='density',
    color='room_type',
    color_discrete_map=palette,
    labels={'price': 'Nightly Price (£)', 'density': 'Density', 'room_type': 'Room Type'},
    title='Entire homes are right-skewed and expensive — private rooms cluster at lower prices',
)

fig.update_traces(fill='tozeroy', opacity=0.4)

fig.add_vline(
    x=med_entire, line_dash='dash', line_color='#2E75B6', line_width=1.5,
    annotation=dict(
        text=f'Entire median £{med_entire:.0f}',
        font=dict(color='#2E75B6', size=11),
        xanchor='left', yanchor='top', xshift=8,
    ),
)

fig.add_vline(
    x=med_private, line_dash='dash', line_color='#E07B39', line_width=1.5,
    annotation=dict(
        text=f'Private median £{med_private:.0f}',
        font=dict(color='#E07B39', size=11),
        xanchor='right', yanchor='top', xshift=-8,
    ),
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(title='Room Type', orientation='h', y=1.08),
    margin=dict(l=60, r=40, t=70, b=40),
)

fig.update_xaxes(gridcolor='#EEEEEE')
fig.update_yaxes(gridcolor='#EEEEEE')

fig.show(renderer="browser")